# RVC Weight Difference & Copy (Element-Level)

1. Computes weight differences between model A and B at the element level
2. Selects TOP X% of elements with the largest differences (using RATIO)
3. Copies these weights from model A to model B

In [190]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from collections import OrderedDict
from typing import List, Dict, Tuple
import datetime

print(f"PyTorch: {torch.__version__}")

PyTorch: 2.3.1


In [ ]:
# ===== CONFIGURATION =====

# Two models for comparison
# Weights are copied from A to B
MODEL_A = "Ravdess_F_Angry_60e_1140s.pth"
MODEL_B = "Samuel.pth"

# Ratio of weights to copy (0.0-1.0)
# E.g. 0.1 = 10% most different weights, 0.01 = 1%
RATIO = 0.5

# Limit to specific modules? (None = all)
# Options: "enc_p", "dec", "flow", "emb_g"
MODULES = ["emb_g"]  # or e.g. ["dec", "flow"]

# Output file
OUTPUT_PATH = "emb_g_05.pth"

In [ ]:
def load_checkpoint(path: str) -> dict:
    """Load checkpoint from .pth file."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"Model not found: {path}")
    cpt = torch.load(path, map_location="cpu")
    print(f"Loaded: {os.path.basename(path)} (SR: {cpt.get('sr')}, v: {cpt.get('version')})")
    return cpt


def extract_weights(cpt: dict) -> OrderedDict:
    """Extract weights from checkpoint (filters enc_q)."""
    weights = cpt.get("model") or cpt.get("weight")
    if weights is None:
        raise ValueError("Unknown checkpoint format")
    return OrderedDict((k, v) for k, v in weights.items() if "enc_q" not in k)


def compute_all_differences(
    cpt_a: dict, 
    cpt_b: dict,
    modules: List[str] = None
) -> Tuple[Dict[str, torch.Tensor], Dict[str, dict]]:
    """
    Compute weight differences between two models at the element level.
    
    Returns:
        diff_tensors: Dict {layer_name: abs_diff_tensor}
        stats: Dict {layer_name: {"mean", "max", "numel", "shape"}}
    """
    w_a = extract_weights(cpt_a)
    w_b = extract_weights(cpt_b)
    
    diff_tensors = {}
    stats = {}
    
    for key in w_a.keys():
        # Module filter
        if modules is not None:
            if not any(key.startswith(m) for m in modules):
                continue
        
        if key not in w_b:
            continue
            
        a = w_a[key].float()
        b = w_b[key].float()
        
        if a.shape != b.shape:
            continue
        
        diff = (a - b).abs()
        diff_tensors[key] = diff
        
        stats[key] = {
            "mean": diff.mean().item(),
            "max": diff.max().item(),
            "numel": a.numel(),
            "shape": list(a.shape)
        }
    
    return diff_tensors, stats


def compute_threshold(diff_tensors: Dict[str, torch.Tensor], ratio: float) -> float:
    """
    Compute threshold for selecting TOP ratio% weights.
    """
    # Concatenate all differences into one vector
    all_diffs = torch.cat([d.flatten() for d in diff_tensors.values()])
    total = all_diffs.numel()
    
    # Number of weights to select
    k = int(total * ratio)
    k = max(1, min(k, total))  # At least 1, max all
    
    # Find the k-th largest element (threshold)
    threshold = torch.kthvalue(all_diffs, total - k + 1).values.item()
    
    print(f"Total parameters: {total:,}")
    print(f"Ratio: {ratio} -> selecting {k:,} weights ({100*ratio:.2f}%)")
    print(f"Threshold: {threshold:.8f}")
    
    return threshold


def create_copy_masks(
    diff_tensors: Dict[str, torch.Tensor], 
    threshold: float
) -> Tuple[Dict[str, torch.Tensor], Dict[str, int]]:
    """
    Create binary masks for weight copying.
    
    Returns:
        masks: Dict {layer_name: bool_tensor} where True = copy
        counts: Dict {layer_name: number_of_selected_elements}
    """
    masks = {}
    counts = {}
    
    for key, diff in diff_tensors.items():
        mask = diff >= threshold
        masks[key] = mask
        counts[key] = mask.sum().item()
    
    total_selected = sum(counts.values())
    print(f"Total selected weights: {total_selected:,}")
    
    return masks, counts


def apply_mask_copy(
    source_cpt: dict,
    target_cpt: dict,
    masks: Dict[str, torch.Tensor]
) -> OrderedDict:
    """
    Copy selected weights (according to masks) from source to target model.
    """
    source_weights = extract_weights(source_cpt)
    target_weights = extract_weights(target_cpt)
    
    result = OrderedDict()
    copied_total = 0
    
    for key in target_weights.keys():
        tgt = target_weights[key].float()
        
        if key in masks and key in source_weights:
            src = source_weights[key].float()
            mask = masks[key]
            
            if src.shape == tgt.shape:
                # Use torch.where for efficient copying
                result[key] = torch.where(mask, src, tgt)
                copied_total += mask.sum().item()
            else:
                result[key] = tgt
        else:
            result[key] = tgt
    
    print(f"Copied {copied_total:,} individual weights")
    return result


def save_model(weights: OrderedDict, base_cpt: dict, output_path: str, info: str = "") -> str:
    """Save model."""
    opt = OrderedDict()
    opt["weight"] = OrderedDict((k, v.half()) for k, v in weights.items())
    opt["config"] = base_cpt.get("config")
    opt["sr"] = base_cpt.get("sr")
    opt["f0"] = base_cpt.get("f0", 1)
    opt["version"] = base_cpt.get("version", "v2")
    opt["creation_date"] = datetime.datetime.now().isoformat()
    opt["info"] = info
    
    torch.save(opt, output_path)
    print(f"Saved: {output_path} ({os.path.getsize(output_path) / 1024 / 1024:.2f} MB)")
    return output_path


print("Functions ready.")

In [ ]:
# ===== 1. LOAD MODELS =====

cpt_a = load_checkpoint(MODEL_A)
cpt_b = load_checkpoint(MODEL_B)

# ===== 2. COMPUTE DIFFERENCES AT ELEMENT LEVEL =====

print("\nComputing weight differences at element level...")
diff_tensors, stats = compute_all_differences(cpt_a, cpt_b, MODULES)
print(f"Analyzed {len(diff_tensors)} layers")

# ===== 3. COMPUTE THRESHOLD AND MASKS =====

print(f"\nComputing threshold for RATIO={RATIO}...")
threshold = compute_threshold(diff_tensors, RATIO)

print("\nCreating masks for copying...")
masks, element_counts = create_copy_masks(diff_tensors, threshold)

In [ ]:
# ===== VISUALIZATION =====

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# --- Chart 1: Histogram of all differences ---
ax1 = axes[0, 0]
all_diffs = torch.cat([d.flatten() for d in diff_tensors.values()]).numpy()
ax1.hist(all_diffs, bins=100, color='steelblue', edgecolor='none', alpha=0.7, log=True)
ax1.axvline(x=threshold, color='red', linestyle='--', linewidth=2, 
            label=f'Threshold: {threshold:.6f}')
ax1.set_xlabel('Absolute Difference')
ax1.set_ylabel('Weight Count (log)')
ax1.set_title(f'Distribution of All Weight Differences (RATIO={RATIO})')
ax1.legend()

# --- Chart 2: TOP layers by number of selected elements ---
ax2 = axes[0, 1]
sorted_counts = sorted(element_counts.items(), key=lambda x: x[1], reverse=True)[:15]
layers_short = [l[:30] + "..." if len(l) > 30 else l for l, _ in sorted_counts]
counts_vals = [c for _, c in sorted_counts]
ax2.barh(range(len(layers_short)), counts_vals, color='coral')
ax2.set_yticks(range(len(layers_short)))
ax2.set_yticklabels(layers_short, fontsize=9)
ax2.set_xlabel('Number of Selected Weights')
ax2.set_title('TOP 15 Layers with Most Selected Weights')
ax2.invert_yaxis()

# --- Chart 3: Distribution by modules ---
ax3 = axes[1, 0]
module_counts = {"enc_p": 0, "dec": 0, "flow": 0, "emb_g": 0, "other": 0}
module_totals = {"enc_p": 0, "dec": 0, "flow": 0, "emb_g": 0, "other": 0}

for layer, count in element_counts.items():
    matched = False
    for m in ["enc_p", "dec", "flow", "emb_g"]:
        if layer.startswith(m):
            module_counts[m] += count
            module_totals[m] += stats[layer]["numel"]
            matched = True
            break
    if not matched:
        module_counts["other"] += count
        module_totals["other"] += stats[layer]["numel"]

# Only modules with data
modules_with_data = [(m, c, module_totals[m]) for m, c in module_counts.items() if module_totals[m] > 0]
mod_names = [m for m, _, _ in modules_with_data]
mod_selected = [c for _, c, _ in modules_with_data]
mod_total = [t for _, _, t in modules_with_data]
mod_pct = [100 * s / t if t > 0 else 0 for s, t in zip(mod_selected, mod_total)]

x = np.arange(len(mod_names))
width = 0.35
bars1 = ax3.bar(x - width/2, mod_selected, width, label='Selected', color='coral')
bars2 = ax3.bar(x + width/2, mod_total, width, label='Total', color='steelblue', alpha=0.5)
ax3.set_ylabel('Weight Count')
ax3.set_title('Distribution of Selected Weights by Module')
ax3.set_xticks(x)
ax3.set_xticklabels(mod_names)
ax3.legend()
ax3.set_yscale('log')

# Add percentages above bars
for i, (rect, pct) in enumerate(zip(bars1, mod_pct)):
    ax3.annotate(f'{pct:.1f}%', xy=(rect.get_x() + rect.get_width()/2, rect.get_height()),
                 xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=9)

# --- Chart 4: Heatmap for layer with most selected weights ---
ax4 = axes[1, 1]
top_layer = sorted_counts[0][0] if sorted_counts else None
if top_layer and len(diff_tensors[top_layer].shape) >= 2:
    diff_2d = diff_tensors[top_layer]
    # If more than 2D, average other dimensions
    while len(diff_2d.shape) > 2:
        diff_2d = diff_2d.mean(dim=0)
    
    # Limit size for display
    max_size = 100
    if diff_2d.shape[0] > max_size:
        diff_2d = diff_2d[:max_size, :]
    if diff_2d.shape[1] > max_size:
        diff_2d = diff_2d[:, :max_size]
    
    im = ax4.imshow(diff_2d.numpy(), aspect='auto', cmap='hot')
    plt.colorbar(im, ax=ax4, label='Abs Difference')
    ax4.set_title(f'Difference Heatmap: {top_layer[:40]}...')
    ax4.set_xlabel('Dimension 1')
    ax4.set_ylabel('Dimension 0')
else:
    ax4.text(0.5, 0.5, 'N/A (1D tensor)', ha='center', va='center', transform=ax4.transAxes)
    ax4.set_title('Difference Heatmap')

plt.tight_layout()
plt.show()

# Statistics table
print(f"\nStatistics:")
print(f"  Total layers: {len(diff_tensors)}")
print(f"  Total parameters: {sum(s['numel'] for s in stats.values()):,}")
print(f"  Selected weights: {sum(element_counts.values()):,} ({100*RATIO:.2f}%)")
print(f"  Threshold: {threshold:.8f}")

In [ ]:
# ===== 4. COPY WEIGHTS =====

print(f"Copying selected weights from model A to model B")
print(f"  Source: {MODEL_A}")
print(f"  Target: {MODEL_B}")
print(f"  Number of weights to copy: {sum(element_counts.values()):,}")

# Copy using masks (from A to B)
result_weights = apply_mask_copy(cpt_a, cpt_b, masks)

# Save
info = f"Copied {RATIO*100:.1f}% most different weights from {os.path.basename(MODEL_A)} into {os.path.basename(MODEL_B)}"
save_model(result_weights, cpt_b, OUTPUT_PATH, info)

print(f"\nDone!")

In [ ]:
# ===== DETAILED VIEW OF LAYERS =====

print(f"Layers with most selected weights:")
print("=" * 80)

sorted_by_count = sorted(element_counts.items(), key=lambda x: x[1], reverse=True)

for i, (layer, count) in enumerate(sorted_by_count[:20], 1):
    total = stats[layer]["numel"]
    pct = 100 * count / total if total > 0 else 0
    print(f"{i:3}. {layer}")
    print(f"     shape: {stats[layer]['shape']}, params: {total:,}")
    print(f"     selected weights: {count:,} ({pct:.2f}%)")
    print(f"     mean diff: {stats[layer]['mean']:.6f}, max diff: {stats[layer]['max']:.6f}")